### Ml-1M

In [ ]:
import os
import pandas as pd
import numpy as np
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# API 키 확인
API_KEY = os.environ.get("OPENAI_API_KEY") 
if not API_KEY:
    raise ValueError("OPENAI_API_KEY environment variable is not set.")

# ========= Configuration =========
# ML-1M 경로 설정
file_path = "./data/ml-1m/ml-1m_llmrec_format"
input_csv = os.path.join(file_path, "item_attributes.csv")

# 모델 설정 (text-embedding-3-small)
embedding_model = "text-embedding-3-small"

# 배치 및 스레드 설정
BATCH_SIZE = 100  # text-embedding-3-small은 비용이 저렴하고 토큰 한도가 넉넉하여 100~200도 무방함
MAX_WORKERS = 10  

# text-embedding-3-small의 기본 차원은 1536입니다.
EMBEDDING_DIM = 1536 

# ========= Load item metadata =========
print(f"📂 Loading CSV from {input_csv}...")

# 이전 단계에서 생성한 csv는 헤더가 없고 순서가 id, year, title, genre 였음
df = pd.read_csv(input_csv, header=None, names=["id", "year", "title", "genre"])

text_repr = []

print("📝 Constructing text representations...")
for i, row in df.iterrows():
    # 데이터 결측치 방어 로직
    title = str(row['title']) if pd.notnull(row['title']) else "Unknown Title"
    year = str(int(row['year'])) if pd.notnull(row['year']) and row['year'] != 0 else "Unknown Year"
    genre = str(row['genre']) if pd.notnull(row['genre']) else "Unknown Genre"
    
    # 영화에 맞는 프롬프트 구성
    # 예: "Title: Toy Story (1995), Year: 1995, Genre: ['Animation', 'Children's', 'Comedy']"
    summary = f"Title: {title}, Year: {year}, Genre: {genre}"
    text_repr.append(summary)

total_items = len(text_repr)
print(f"✅ Total items to embed: {total_items}")

# ========= Worker Function =========
def get_embeddings_batch(start_idx, batch_texts):
    """
    배치 단위로 임베딩을 요청하는 작업자 함수
    """
    url = "https://api.openai.com/v1/embeddings"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": embedding_model,
        "input": batch_texts
    }
    
    # 재시도 로직 (최대 3회)
    for attempt in range(3):
        try:
            response = requests.post(url, headers=headers, json=data, timeout=30)
            
            if response.status_code == 200:
                res_json = response.json()
                embeddings = [item['embedding'] for item in res_json['data']]
                return start_idx, embeddings, None  # 성공
            
            elif response.status_code >= 500: # 서버 에러
                time.sleep(1 * (attempt + 1))
                continue
            elif response.status_code == 429: # Rate Limit
                time.sleep(2 * (attempt + 1))
                continue
            else:
                # 400 Bad Request 등
                error_msg = f"HTTP {response.status_code}: {response.text}"
                break
                
        except Exception as e:
            time.sleep(1)
            error_msg = str(e)
            
    # 최종 실패 시
    print(f"❌ Batch failed ({start_idx}~): {error_msg}")
    # 실패한 개수만큼 0 벡터 채움
    zero_vectors = [np.zeros(EMBEDDING_DIM) for _ in range(len(batch_texts))]
    return start_idx, zero_vectors, start_idx  # 실패한 배치의 시작 인덱스 반환

# ========= Main Parallel Process =========
def main():
    print(f"🚀 Starting Parallel Embedding Generation using {embedding_model}...")

    results = []
    failed_indices_log = []

    # 데이터 배칭
    batches = []
    for i in range(0, total_items, BATCH_SIZE):
        batch_texts = text_repr[i : i + BATCH_SIZE]
        batches.append((i, batch_texts))

    # 병렬 처리 시작
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(get_embeddings_batch, idx, txts) for idx, txts in batches]
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Embedding Batches"):
            start_idx, embs, failed_idx = future.result()
            results.append((start_idx, embs))
            
            if failed_idx is not None:
                batch_len = len(embs)
                failed_indices_log.extend(range(failed_idx, failed_idx + batch_len))

    # ========= Reassemble & Save =========
    print("📦 Reassembling results...")
    results.sort(key=lambda x: x[0])

    final_embeddings = []
    for _, embs in results:
        final_embeddings.extend(embs)

    # numpy 변환
    embedding_array = np.array(final_embeddings)
    print(f"📊 Final Shape: {embedding_array.shape}")

    # 저장
    output_path = os.path.join(file_path, "text_feat.npy")
    np.save(output_path, embedding_array)
    print(f"✅ Saved text_feat.npy to {output_path}")

    if failed_indices_log:
        print(f"⚠️ Total failed items: {len(failed_indices_log)}")
        print(f"Failed indices samples: {failed_indices_log[:10]} ...")
    else:
        print("🎉 All items processed successfully!")

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import pickle
import os

# 설정
file_path = "./data/ml-1m/ml-1m_llmrec_format"
npy_path = os.path.join(file_path, "image_feat.npy")
embedding_dim = 512  # 원하는 차원
train_mat_path = os.path.join(file_path, "train_mat")

# 아이템 수 확인
with open(train_mat_path, "rb") as f:
    train_mat = pickle.load(f)
n_items = train_mat.shape[1]
print(f"✅ Number of items: {n_items}")

# zero vector 생성
image_feat = np.zeros((n_items, embedding_dim), dtype=np.float32)
np.save(npy_path, image_feat)

print(f"✅ Saved zero image_feat.npy to {npy_path}")
print(f"Shape: {image_feat.shape}")


### Books 


In [ ]:
import os
import pandas as pd
import numpy as np
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

API_KEY = os.environ.get("OPENAI_API_KEY") 

# ========= Configuration =========
file_path = "./data/books/books_llmrec_format"
input_csv = os.path.join(file_path, "item_attribute.csv")
embedding_model = "text-embedding-ada-002"
BATCH_SIZE = 100  # 한 번 요청에 묶을 아이템 수 (너무 크면 토큰 제한 걸릴 수 있음)
MAX_WORKERS = 10  # 병렬 스레드 개수

# ========= Load item metadata =========
print("📂 Loading CSV...")
# header=None이므로 names로 컬럼 지정
df = pd.read_csv(input_csv, names=["id", "brand", "title", "category"])
text_repr = []

for i, row in df.iterrows():
    brand = str(row['brand']) if pd.notnull(row['brand']) else "unknown"
    title = str(row['title']) if pd.notnull(row['title']) else "unknown"
    category = str(row['category']) if pd.notnull(row['category']) else "unknown"
    summary = f"{brand}, {title}, {category}"
    text_repr.append(summary)

total_items = len(text_repr)
print(f"✅ Total items to embed: {total_items}")

# ========= Worker Function =========
def get_embeddings_batch(start_idx, batch_texts):
    """
    배치 단위로 임베딩을 요청하는 작업자 함수
    """
    url = "https://api.openai.com/v1/embeddings"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": embedding_model,
        "input": batch_texts  # 리스트 통째로 전달
    }
    
    # 재시도 로직 (최대 3회)
    for attempt in range(3):
        try:
            response = requests.post(url, headers=headers, json=data, timeout=20)
            
            if response.status_code == 200:
                res_json = response.json()
                # 반환된 데이터에서 embedding 추출 (순서대로 옴)
                embeddings = [item['embedding'] for item in res_json['data']]
                return start_idx, embeddings, None  # 성공
            
            elif response.status_code >= 500: # 서버 에러는 잠시 대기 후 재시도
                time.sleep(1 * (attempt + 1))
                continue
            else:
                # 4xx 에러 등은 즉시 실패 처리
                error_msg = f"HTTP {response.status_code}: {response.text}"
                break
                
        except Exception as e:
            time.sleep(1)
            error_msg = str(e)
            
    # 최종 실패 시
    print(f"❌ Batch failed ({start_idx}~): {error_msg}")
    # 실패한 개수만큼 0 벡터 채움
    zero_vectors = [np.zeros(1536) for _ in range(len(batch_texts))]
    return start_idx, zero_vectors, start_idx  # 실패한 배치의 시작 인덱스 반환

# ========= Main Parallel Process =========
print("🚀 Starting Parallel Embedding Generation...")

# 결과를 담을 리스트 (인덱스 순서대로 나중에 정렬하기 위해 튜플 저장)
results = []
failed_indices_log = []

# 데이터 배칭 (Chunking)
batches = []
for i in range(0, total_items, BATCH_SIZE):
    batch_texts = text_repr[i : i + BATCH_SIZE]
    batches.append((i, batch_texts))

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Future 생성
    futures = [executor.submit(get_embeddings_batch, idx, txts) for idx, txts in batches]
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Embedding Batches"):
        start_idx, embs, failed_idx = future.result()
        results.append((start_idx, embs))
        
        if failed_idx is not None:
            # 실패한 배치의 인덱스들 기록
            batch_len = len(embs)
            failed_indices_log.extend(range(failed_idx, failed_idx + batch_len))

# ========= Reassemble & Save =========
print("📦 Reassembling results...")
# start_idx 기준으로 정렬하여 순서 맞춤
results.sort(key=lambda x: x[0])

# 리스트 flatten (2차원 리스트 -> 1차원 리스트)
final_embeddings = []
for _, embs in results:
    final_embeddings.extend(embs)

# numpy 변환
embedding_array = np.array(final_embeddings)
print(f"📊 Final Shape: {embedding_array.shape}")

# 저장
np.save(os.path.join(file_path, "text_feat.npy"), embedding_array)
print(f"✅ Saved text_feat.npy to {file_path}")

if failed_indices_log:
    print(f"⚠️ Total failed items: {len(failed_indices_log)}")
    print(f"Failed indices samples: {failed_indices_log[:10]} ...")
else:
    print("🎉 All items processed successfully!")

In [ ]:
import pickle
import os
import json

def check_profiling_dict(dataset="books"):
    # 1. 경로 설정 (본인 환경에 맞게 수정)
    if dataset == "netflix":
        file_path = "./LLMRec/LLMRec_c/" + dataset + "/netflix_valid_item/"
    elif dataset == "movielens":
        file_path = "./LLMRec/LLMRec_c/" + dataset + "/ml100k_llmrec_format/"
    elif dataset == "books":
        file_path = "./data/books/books_llmrec_format/"
    elif dataset == "mind":
        file_path = "./data/mind/mind_llmrec_format/"
    
    dict_path = os.path.join(file_path, "augmented_user_profiling_dict")

    # 2. 파일 존재 확인
    if not os.path.exists(dict_path):
        print(f"❌ 파일이 존재하지 않습니다: {dict_path}")
        return

    # 3. Pickle 로드
    print(f"📂 Loading: {dict_path} ...")
    with open(dict_path, 'rb') as f:
        data = pickle.load(f)

    # 4. 기본 정보 출력
    print(f"✅ Total Users Processed: {len(data)}")
    
    # 5. 샘플 데이터 출력 (앞에서 3개만)
    print("\n🔍 Sample Profiles (First 3 users):")
    print("="*50)
    
    cnt = 0
    for user_id, profile_content in data.items():
        print(f"👤 [User ID: {user_id}]")
        
        # JSON 파싱 시도 (보기 좋게 출력하기 위함)
        try:
            # LLM이 가끔 ```json ... ``` 태그를 붙이는 경우가 있어 제거 시도
            clean_content = profile_content.replace("```json", "").replace("```", "").strip()
            profile_json = json.loads(clean_content)
            print(json.dumps(profile_json, indent=4, ensure_ascii=False))
        except json.JSONDecodeError:
            # 파싱 실패시 원본 텍스트 출력
            print("⚠️ (Raw Text - Not Valid JSON)")
            print(profile_content)
        
        print("-" * 50)
        
        cnt += 1
        if cnt >= 5: # 3명만 보고 종료
            break

if __name__ == "__main__":
    # 확인하고 싶은 데이터셋 이름 입력 ("books", "movielens", "netflix")
    check_profiling_dict("books")

In [ ]:
import pickle
import os

# 파일 경로 설정
file_path = './data/books/books_llmrec_format/augmented_sample_dict'

# 1. 파일 존재 여부 확인
if not os.path.exists(file_path):
    print(f"❌ Error: 파일이 존재하지 않습니다.\n경로를 확인하세요: {file_path}")
else:
    try:
        # 2. 파일 로드
        with open(file_path, 'rb') as f:
            augmented_sample_dict = pickle.load(f)

        # 3. 데이터 개수 확인
        print(f"✅ 로드 성공! 총 유저 수: {len(augmented_sample_dict)}")
        augmented_sample_dict = dict(sorted(augmented_sample_dict.items(), key=lambda item: item[0])) 
        # 4. 내용 확인 (앞에서 5개만 출력)
        if len(augmented_sample_dict) == 0:
            print("⚠️ 딕셔너리가 비어있습니다.")
        else:
            print("\n🔍 앞부분 데이터 샘플 (5개):")
            count = 0
            for user_id, content in augmented_sample_dict.items():
                print(f"User {user_id}: {content}")
                count += 1
                if count >= 5:
                    break

    except EOFError:
        print("❌ Error: 파일이 비어있거나 손상되었습니다. (저장이 덜 되었을 수 있습니다)")
    except Exception as e:
        print(f"❌ 알 수 없는 에러 발생: {e}")

### MIND


In [ ]:
import os
import pandas as pd
import numpy as np
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# ========= Configuration =========
API_KEY = os.environ.get("OPENAI_API_KEY")
if not API_KEY:
    raise ValueError("OPENAI_API_KEY environment variable is not set.")

EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536 
BATCH_SIZE = 100  # API 호출 1회당 묶어서 보낼 텍스트 수
MAX_WORKERS = 10  

# ========= Helper Function =========
def construct_text_repr(row, dataset):
    """데이터셋 종류에 맞춰 임베딩할 텍스트(문장)를 생성합니다."""
    ds = dataset.lower()
    
    # 헬퍼 함수: 결측치(NaN) 안전 처리
    def get_val(key, is_year=False):
        val = row.get(key)
        if pd.isnull(val) or val == "":
            return "Unknown"
        if is_year:
            try:
                return str(int(val)) if int(val) != 0 else "Unknown"
            except ValueError:
                return "Unknown"
        return str(val)

    if ds == "netflix":
        return f"Title: {get_val('title')}, Year: {get_val('year', is_year=True)}"
    
    elif ds == "movielens":
        return f"Title: {get_val('title')}, Year: {get_val('year', is_year=True)}, Genre: {get_val('genre')}"
    
    elif ds == "books":
        return f"Brand: {get_val('brand')}, Title: {get_val('title')}, Category: {get_val('category')}"
    
    elif ds == "mind":
        return f"Title: {get_val('title')}, Category: {get_val('category')}, Subcategory: {get_val('subcategory')}"
    
    raise ValueError(f"Unknown dataset: {dataset}")

# ========= Worker Function =========
def get_embeddings_batch(start_idx, batch_texts):
    """배치 단위로 임베딩을 요청하는 작업자 함수"""
    url = "https://api.openai.com/v1/embeddings"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": EMBEDDING_MODEL,
        "input": batch_texts
    }
    
    error_msg = "Unknown Error"
    
    for attempt in range(3):
        try:
            response = requests.post(url, headers=headers, json=data, timeout=30)
            
            if response.status_code == 200:
                res_json = response.json()
                # 성공 시 임베딩 리스트 반환
                embeddings = [item['embedding'] for item in res_json['data']]
                return start_idx, embeddings, None
            
            elif response.status_code in [429, 500, 502, 503, 504]:
                time.sleep(2 * (attempt + 1))
                continue
            else:
                error_msg = f"HTTP {response.status_code}: {response.text}"
                break
                
        except Exception as e:
            error_msg = str(e)
            time.sleep(2)
            
    print(f"❌ Batch failed ({start_idx}~{start_idx+len(batch_texts)-1}): {error_msg}")
    zero_vectors = [np.zeros(EMBEDDING_DIM) for _ in range(len(batch_texts))]
    return start_idx, zero_vectors, start_idx

# ========= Main Process =========
def main(dataset="mind"):
    print(f"🚀 Starting Parallel Embedding Generation for [{dataset.upper()}]...")

    # 데이터셋별 경로 및 컬럼 설정
    if dataset == "netflix":
        file_path = "./LLMRec/LLMRec_c/netflix/netflix_valid_item"
        cols = ["id", "year", "title"]
    elif dataset == "movielens":
        file_path = "./data/ml-1m/ml-1m_llmrec_format"
        cols = ["id", "year", "title", "genre"]
    elif dataset == "books":
        file_path = "./data/books/books_llmrec_format"
        cols = ["id", "brand", "title", "category"]
    elif dataset == "mind":
        file_path = "./data/mind/mind_llmrec_format"
        cols = ["id", "title", "category", "subcategory"]
    else:
        raise ValueError(f"Unknown dataset: {dataset}")

    input_csv = os.path.join(file_path, "item_attribute.csv") # 원본 코드 item_attributes.csv 였으나 기존 컨벤션 통일
    output_npy = os.path.join(file_path, "text_feat.npy")

    # 1. 데이터 로드
    print(f"📂 Loading CSV from {input_csv}...")
    df = pd.read_csv(input_csv, header=None, names=cols)

    # 2. 텍스트 변환
    print("📝 Constructing text representations...")
    text_repr = [construct_text_repr(row, dataset) for _, row in df.iterrows()]
    total_items = len(text_repr)
    print(f"✅ Total items to embed: {total_items}")

    # 3. 데이터 배칭 (OpenAI Embedding API는 배열로 묶어서 보내는 것이 효율적)
    batches = []
    for i in range(0, total_items, BATCH_SIZE):
        batches.append((i, text_repr[i : i + BATCH_SIZE]))

    results = []
    failed_indices_log = []

    # 4. 병렬 처리 시작
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(get_embeddings_batch, idx, txts) for idx, txts in batches]
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Embedding Batches"):
            start_idx, embs, failed_idx = future.result()
            results.append((start_idx, embs))
            
            if failed_idx is not None:
                failed_indices_log.extend(range(failed_idx, failed_idx + len(embs)))

    # 5. 결과 조립 및 저장
    print("📦 Reassembling results...")
    results.sort(key=lambda x: x[0]) # 비동기 처리된 결과를 원래 인덱스 순서대로 정렬

    final_embeddings = []
    for _, embs in results:
        final_embeddings.extend(embs)

    embedding_array = np.array(final_embeddings)
    print(f"📊 Final Shape: {embedding_array.shape}")

    np.save(output_npy, embedding_array)
    print(f"✅ Saved text_feat.npy to {output_npy}")

    if failed_indices_log:
        print(f"⚠️ Total failed items: {len(failed_indices_log)}")
        print(f"Failed indices samples: {failed_indices_log[:10]} ...")
    else:
        print("🎉 All items processed successfully!")

if __name__ == "__main__":
    # 실행할 데이터셋 이름만 바꿔주면 완벽히 호환됩니다.
    main("mind")

In [ ]:
import numpy as np
import pickle
import os

# 설정
file_path = "./data/yelp/yelp_llmrec_format"
npy_path = os.path.join(file_path, "image_feat.npy")
embedding_dim = 512  # 원하는 차원
train_mat_path = os.path.join(file_path, "train_mat")

# 아이템 수 확인
with open(train_mat_path, "rb") as f:
    train_mat = pickle.load(f)
n_items = train_mat.shape[1]
print(f"✅ Number of items: {n_items}")

# zero vector 생성
image_feat = np.zeros((n_items, embedding_dim), dtype=np.float32)
np.save(npy_path, image_feat)

print(f"✅ Saved zero image_feat.npy to {npy_path}")
print(f"Shape: {image_feat.shape}")
